In [1]:
from itertools import product


def solve_knapsack(weights, values, capacity):
    if len(weights) != len(values):
        raise ValueError("Weights and values must have equal lengths.")
    if capacity < 0 or any(w < 0 for w in weights):
        raise ValueError("Capacity and weights must be non-negative.")
    if any(v < 0 for v in values):
        raise ValueError("Values must be non-negative.")

    best_value = 0
    best_weight = 0
    best_selection = [0] * len(weights)

    for selection in product([0, 1], repeat=len(weights)):
        total_weight = sum(
            selection[i] * weights[i] for i in range(len(weights))
        )
        total_value = sum(
            selection[i] * values[i] for i in range(len(values))
        )

        if total_weight <= capacity and total_value > best_value:
            best_value = total_value
            best_weight = total_weight
            best_selection = list(selection)

    return {
        "selection": best_selection,
        "total_weight": best_weight,
        "total_value": best_value,
    }


result = solve_knapsack([3, 4, 2], [4, 5, 3], 7)
print(result)

assert result["selection"] == [1, 1, 0]
assert result["total_weight"] == 7
assert result["total_value"] == 9

print("All basic knapsack checks passed!")

{'selection': [1, 1, 0], 'total_weight': 7, 'total_value': 9}
All basic knapsack checks passed!


In [2]:
%pip install -q qiskit qiskit-optimization

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.0/664.0 kB 3.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.1/237.1 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 2.4 MB/s eta 0:00:00


In [3]:
import qiskit
import qiskit_optimization

print("Qiskit version:", qiskit.__version__)
print("Qiskit Optimization imported successfully!")

Qiskit version: 2.5.2
Qiskit Optimization imported successfully!


In [4]:
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.converters import QuadraticProgramToQubo

# Small example for testing our quantum formulation
weights = [3, 4, 2]
item_values = [4, 5, 3]
capacity = 7

# Define the 0-1 knapsack problem
problem = QuadraticProgram("knapsack")

for i in range(len(weights)):
    problem.binary_var(name=f"x{i}")

problem.maximize(
    linear={
        f"x{i}": item_values[i]
        for i in range(len(item_values))
    }
)

problem.linear_constraint(
    linear={
        f"x{i}": weights[i]
        for i in range(len(weights))
    },
    sense="<=",
    rhs=capacity,
    name="capacity",
)

print(problem.prettyprint())

# Convert the constrained problem to a QUBO
converter = QuadraticProgramToQubo()
qubo = converter.convert(problem)

print("\nQUBO formulation:")
print(qubo.prettyprint())

# Convert QUBO into an Ising operator
ising_operator, offset = qubo.to_ising()

print("\nIsing operator:")
print(ising_operator)
print("Energy offset:", offset)
print("Number of qubits:", ising_operator.num_qubits)

Problem name: knapsack

Maximize
  4*x0 + 5*x1 + 3*x2

Subject to
  Linear constraints (1)
    3*x0 + 4*x1 + 2*x2 <= 7  'capacity'

  Binary variables (3)
    x0 x1 x2


QUBO formulation:
Problem name: knapsack

Minimize
  13*capacity@int_slack@0^2 + 52*capacity@int_slack@0*capacity@int_slack@1
  + 104*capacity@int_slack@0*capacity@int_slack@2 + 52*capacity@int_slack@1^2
  + 208*capacity@int_slack@1*capacity@int_slack@2 + 208*capacity@int_slack@2^2
  + 78*x0*capacity@int_slack@0 + 156*x0*capacity@int_slack@1
  + 312*x0*capacity@int_slack@2 + 117*x0^2 + 312*x0*x1 + 156*x0*x2
  + 104*x1*capacity@int_slack@0 + 208*x1*capacity@int_slack@1
  + 416*x1*capacity@int_slack@2 + 208*x1^2 + 208*x1*x2
  + 52*x2*capacity@int_slack@0 + 104*x2*capacity@int_slack@1
  + 208*x2*capacity@int_slack@2 + 52*x2^2 - 182*capacity@int_slack@0
  - 364*capacity@int_slack@1 - 728*capacity@int_slack@2 - 550*x0 - 733*x1
  - 367*x2 + 637

Subject to
  No constraints

  Binary variables (6)
    x0 x1 x2 capacity@int_sl

In [5]:
from qiskit.circuit.library import QAOAAnsatz

qaoa_circuit = QAOAAnsatz(
    cost_operator=ising_operator,
    reps=1
)

print("QAOA circuit created successfully!")
print("Number of qubits:", qaoa_circuit.num_qubits)
print("Number of parameters:", len(qaoa_circuit.parameters))
print("Circuit depth:", qaoa_circuit.depth())

QAOA circuit created successfully!
Number of qubits: 6
Number of parameters: 2
Circuit depth: 1


In [6]:
from qiskit import transpile
from qiskit.primitives import StatevectorSampler

# Bind trial parameters to the QAOA circuit
parameters = list(qaoa_circuit.parameters)

print("QAOA parameters:", parameters)

parameter_values = [0.5, 0.7]

if len(parameters) != len(parameter_values):
    raise ValueError(
        f"Expected 2 parameters, but found {len(parameters)}."
    )

bound_circuit = qaoa_circuit.assign_parameters(
    dict(zip(parameters, parameter_values))
)

# Add measurements
measured_circuit = bound_circuit.copy()
measured_circuit.measure_all()

# Run 1000 shots using a simulator
sampler = StatevectorSampler(seed=42)
job = sampler.run([measured_circuit], shots=1000)
result = job.result()

counts = result[0].data.meas.get_counts()

print("\nMeasured bit strings:")
print(counts)
print("\nTotal shots:", sum(counts.values()))

QAOA parameters: [ParameterVectorElement(β[0]), ParameterVectorElement(γ[0])]


/usr/local/lib/python3.13/dist-packages/scipy/sparse/linalg/_dsolve/linsolve.py:606: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/usr/local/lib/python3.13/dist-packages/scipy/sparse/linalg/_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
/usr/local/lib/python3.13/dist-packages/scipy/sparse/_index.py:168: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])



Measured bit strings:
{'110001': 90, '011111': 67, '110101': 27, '101110': 12, '000110': 27, '111101': 14, '001010': 54, '011001': 8, '111000': 59, '101100': 36, '110011': 1, '001111': 12, '100100': 48, '000101': 32, '110100': 25, '101011': 15, '011000': 19, '111100': 4, '001100': 10, '101101': 18, '111011': 2, '010110': 15, '001011': 21, '010100': 9, '011010': 12, '010010': 44, '000001': 14, '001000': 17, '010011': 5, '000011': 11, '001110': 10, '011011': 24, '110010': 8, '101111': 14, '100010': 7, '111010': 23, '001101': 5, '100101': 1, '100111': 32, '011101': 3, '100001': 12, '000111': 15, '100000': 1, '100011': 18, '000010': 1, '110000': 6, '000100': 5, '010101': 8, '111111': 12, '010001': 19, '110110': 10, '001001': 5, '101000': 2, '010000': 4, '011110': 4, '101001': 9, '000000': 7, '110111': 1, '101010': 3, '010111': 2, '111110': 1}

Total shots: 1000


In [7]:
# Decode measured QUBO samples and check feasibility
samples = converter.interpret(
    qubo,
    # No input here; use the original problem's sample decoder below.
) if False else None

# The converter maps original variables to the QUBO variables.
# Decode each measured bit string using the QUBO variable order.
feasible_counts = {}
infeasible_counts = {}

for bitstring, count in counts.items():
    # Qiskit displays classical bit strings in reverse variable order.
    bits = [int(bit) for bit in bitstring[::-1]]

    # The first three variables represent the original items.
    selection = bits[:len(weights)]

    total_weight = sum(
        selection[i] * weights[i] for i in range(len(weights))
    )
    total_value = sum(
        selection[i] * item_values[i] for i in range(len(item_values))
    )

    record = {
        "selection": selection,
        "weight": total_weight,
        "value": total_value,
        "count": count,
    }

    if total_weight <= capacity:
        feasible_counts[bitstring] = record
    else:
        infeasible_counts[bitstring] = record

print("Feasible measured states:", len(feasible_counts))
print("Infeasible measured states:", len(infeasible_counts))
print("\nFeasible candidates:")
for bitstring, record in feasible_counts.items():
    print(bitstring, record)

Feasible measured states: 53
Infeasible measured states: 8

Feasible candidates:
110001 {'selection': [1, 0, 0], 'weight': 3, 'value': 4, 'count': 90}
110101 {'selection': [1, 0, 1], 'weight': 5, 'value': 7, 'count': 27}
101110 {'selection': [0, 1, 1], 'weight': 6, 'value': 8, 'count': 12}
000110 {'selection': [0, 1, 1], 'weight': 6, 'value': 8, 'count': 27}
111101 {'selection': [1, 0, 1], 'weight': 5, 'value': 7, 'count': 14}
001010 {'selection': [0, 1, 0], 'weight': 4, 'value': 5, 'count': 54}
011001 {'selection': [1, 0, 0], 'weight': 3, 'value': 4, 'count': 8}
111000 {'selection': [0, 0, 0], 'weight': 0, 'value': 0, 'count': 59}
101100 {'selection': [0, 0, 1], 'weight': 2, 'value': 3, 'count': 36}
110011 {'selection': [1, 1, 0], 'weight': 7, 'value': 9, 'count': 1}
100100 {'selection': [0, 0, 1], 'weight': 2, 'value': 3, 'count': 48}
000101 {'selection': [1, 0, 1], 'weight': 5, 'value': 7, 'count': 32}
110100 {'selection': [0, 0, 1], 'weight': 2, 'value': 3, 'count': 25}
101011 {'se

In [8]:
# Print the verified classical optimum for comparison
classical_result = solve_knapsack(weights, item_values, capacity)

print("CLASSICAL OPTIMUM")
print("Selection:", classical_result["selection"])
print("Total weight:", classical_result["total_weight"])
print("Total value:", classical_result["total_value"])

CLASSICAL OPTIMUM
Selection: [1, 1, 0]
Total weight: 7
Total value: 9


In [9]:
# Decode the QUBO samples back into the original knapsack variables
from qiskit_optimization import QuadraticProgram

print("Original variables:", problem.variables)
print("QUBO variables:", qubo.variables)
print("Converter:", type(converter).__name__)

print("\nClassical reference:")
print("Selection:", classical_result["selection"])
print("Weight:", classical_result["total_weight"])
print("Value:", classical_result["total_value"])

Original variables: [<Variable: x0 (binary)>, <Variable: x1 (binary)>, <Variable: x2 (binary)>]
QUBO variables: [<Variable: x0 (binary)>, <Variable: x1 (binary)>, <Variable: x2 (binary)>, <Variable: capacity@int_slack@0 (binary)>, <Variable: capacity@int_slack@1 (binary)>, <Variable: capacity@int_slack@2 (binary)>]
Converter: QuadraticProgramToQubo

Classical reference:
Selection: [1, 1, 0]
Weight: 7
Value: 9


In [10]:
# Convert each measured bitstring into a Qiskit optimization sample.
from qiskit_optimization.algorithms import (
    OptimizationResult,
    OptimizationResultStatus,
)

decoded_samples = []

for bitstring, count in counts.items():
    bits = [int(bit) for bit in bitstring[::-1]]

    # Keep only samples matching the QUBO's variable count.
    if len(bits) != qubo.get_num_vars():
        continue

    decoded = converter.interpret(bits)

    decoded_samples.append({
        "bitstring": bitstring,
        "count": count,
        "decoded_variables": decoded.tolist(),
    })

print("Successfully decoded samples:", len(decoded_samples))
print("Example:", decoded_samples[0] if decoded_samples else "No samples")

Successfully decoded samples: 61
Example: {'bitstring': '110001', 'count': 90, 'decoded_variables': [1.0, 0.0, 0.0]}


In [11]:
print("DECODED QUANTUM SAMPLES")
print("-" * 35)

for sample in decoded_samples[:10]:
    print("Bitstring:", sample["bitstring"])
    print("Measurements:", sample["count"])
    print("Decoded variables:", sample["decoded_variables"])
    print()

print("Total decoded samples:", len(decoded_samples))

DECODED QUANTUM SAMPLES
-----------------------------------
Bitstring: 110001
Measurements: 90
Decoded variables: [1.0, 0.0, 0.0]

Bitstring: 011111
Measurements: 67
Decoded variables: [1.0, 1.0, 1.0]

Bitstring: 110101
Measurements: 27
Decoded variables: [1.0, 0.0, 1.0]

Bitstring: 101110
Measurements: 12
Decoded variables: [0.0, 1.0, 1.0]

Bitstring: 000110
Measurements: 27
Decoded variables: [0.0, 1.0, 1.0]

Bitstring: 111101
Measurements: 14
Decoded variables: [1.0, 0.0, 1.0]

Bitstring: 001010
Measurements: 54
Decoded variables: [0.0, 1.0, 0.0]

Bitstring: 011001
Measurements: 8
Decoded variables: [1.0, 0.0, 0.0]

Bitstring: 111000
Measurements: 59
Decoded variables: [0.0, 0.0, 0.0]

Bitstring: 101100
Measurements: 36
Decoded variables: [0.0, 0.0, 1.0]

Total decoded samples: 61


In [12]:
best_quantum_candidate = None
feasible_candidates = []

for sample in decoded_samples:
    selection = sample["decoded_variables"]

    if len(selection) != len(weights):
        continue

    total_weight = sum(
        selection[i] * weights[i] for i in range(len(weights))
    )
    total_value = sum(
        selection[i] * item_values[i] for i in range(len(item_values))
    )

    if total_weight <= capacity:
        candidate = {
            "selection": selection,
            "weight": total_weight,
            "value": total_value,
            "count": sample["count"],
        }
        feasible_candidates.append(candidate)

        if (
            best_quantum_candidate is None
            or candidate["value"] > best_quantum_candidate["value"]
        ):
            best_quantum_candidate = candidate

print("Distinct feasible candidates:", len(feasible_candidates))

if best_quantum_candidate is not None:
    print("Best feasible candidate:", best_quantum_candidate)
else:
    print("No feasible candidates found.")

print("Classical optimum:", classical_result)

Distinct feasible candidates: 53
Best feasible candidate: {'selection': [1.0, 1.0, 0.0], 'weight': 7.0, 'value': 9.0, 'count': 1}
Classical optimum: {'selection': [1, 1, 0], 'total_weight': 7, 'total_value': 9}


In [13]:
print("QUANTUM CANDIDATE")
print("Selection:", best_quantum_candidate["selection"])
print("Total weight:", best_quantum_candidate["weight"])
print("Total value:", best_quantum_candidate["value"])

print("\nCLASSICAL OPTIMUM")
print("Selection:", classical_result["selection"])
print("Total weight:", classical_result["total_weight"])
print("Total value:", classical_result["total_value"])

assert best_quantum_candidate["weight"] <= capacity
assert best_quantum_candidate["value"] == classical_result["total_value"]

print("\nCandidate matches the classical optimal value!")

QUANTUM CANDIDATE
Selection: [1.0, 1.0, 0.0]
Total weight: 7.0
Total value: 9.0

CLASSICAL OPTIMUM
Selection: [1, 1, 0]
Total weight: 7
Total value: 9

Candidate matches the classical optimal value!
